In [6]:
!python v20240507_generate_h5_data_with_scales.py

No. of Matched Files: 256
No. of Missing Pairs: 0
256it [01:18,  3.25it/s]


In [9]:
%load_ext autoreload
%autoreload 2

In [5]:
from v20240507_generate_h5_data_with_scales import *

In [11]:
# Example usage
backbone_dir = '../data/backbones'
homolog_dir = '../data/full_pdb_homologs_new'
output_file = './data/20240512_cryo_data_with_scales_and_chain.h5'

matched_files, missing_pairs = match_files(backbone_dir, homolog_dir)
print("No. of Matched Files:", len(matched_files))
print("No. of Missing Pairs:", len(missing_pairs))

for backbone_file, homolog_file in tqdm(zip(sorted(os.listdir(backbone_dir)), sorted(os.listdir(homolog_dir)))):
    true_ca_coords = parse_ca_atoms(os.path.join(backbone_dir, backbone_file))
    homolog_ca_coords = parse_ca_atoms(os.path.join(homolog_dir, homolog_file))
    
    # Convert coordinates to binary grids
    # Changing to 64^3 here
    true_scale, true_ca = coords_to_binary_grid(true_ca_coords, (64,64,64))
    homolog_scale, homolog_ca = coords_to_binary_grid(homolog_ca_coords)
    _, true_vol = create_gaussian_volume(true_ca)  
    break

No. of Matched Files: 256
No. of Missing Pairs: 0


0it [00:01, ?it/s]

(8544, 3)
(3,)
(3,)
(8544, 3)
(3,)
(3,)


array([63, 63, 63])

In [38]:
import torch
import numpy as np
from torch.utils.data import Dataset
import h5py

class CryoData(Dataset):
    def __init__(self, h5_file):
        self.h5_file = h5_file
        with h5py.File(self.h5_file, 'r') as file:
            self.keys = list(file.keys())

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        with h5py.File(self.h5_file, 'r') as file:
            key_name = self.keys[idx]
            group = file[key_name]
            true_ca = torch.tensor(group['true_ca'][:])
            homolog_ca = torch.tensor(group['homolog_ca'][:])
            true_vol = torch.tensor(group['true_vol'][:])
            true_scale = torch.tensor(group['true_scale'][:])
            homolog_scale = torch.tensor(group['homolog_scale'][:])
            # scale_factors = torch.tensor(group['scale_factors'][:])
        return {'name': key_name[:4],
                'true_ca': true_ca, 
                'homolog_ca': homolog_ca, 
                'true_vol': true_vol,
                'true_scale': true_scale, 
                'homolog_scale': homolog_scale}

# Usage example
dataset = CryoData('./data/20240507_cryo_data_with_scales.h5')

In [39]:

# Assuming you have a CryoData instance called 'dataset'
for i in range(1):  # Check the first three samples
    sample = dataset[i]
    print(f"Sample {i}:")
    print(f"looking at {sample['name']}")
    print(sample)

Sample 0:
looking at 2Y9J
{'name': '2Y9J', 'true_ca': tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         